# T3 leakage runs on Colab

Runs `t3_leakage/run_leakage.py` (120-sentence corpus, held-out sentences, 3 layers, 5 splits, bootstrap CIs) for each model and downloads the result JSONs.

Before running:
1. Runtime > Change runtime type > GPU. A 24 GB or larger GPU (L4, A100) holds the 7-8B models fully; a T4 works for TinyLlama and spills part of a 7B model to CPU (slower, still fine for this job).
2. In the key icon (Secrets) panel add `HF_TOKEN` (needed for the gated Mistral and Llama repos; accept their licences on huggingface.co first) and, if the GitHub repo is private, `GH_TOKEN`.
3. Push the current repo state to GitHub first: the notebook clones it.

Two run presets (set `PRESET` in the models cell):
- `pi-robust` (default): FP baseline and `turbo_k3v4` only, with the public, naive secret-Pi and Pi-robust (8 attacker rotations) inverters. A few minutes per model plus the download.
- `ci-rerun`: the full config grid (already in `results/t3_leakage/*-ci-rerun/`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
import os
from google.colab import userdata

REPO = "github.com/hassanmahmad/turboquant-exp.git"
BRANCH = "main"

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

gh = secret("GH_TOKEN")
url = f"https://{gh}@{REPO}" if gh else f"https://{REPO}"
!rm -rf /content/turboquant-exp
!git clone -q --branch {BRANCH} --depth 1 {url} /content/turboquant-exp
%cd /content/turboquant-exp
!git log --oneline -1
!pip install -q "transformers>=4.57,<5" accelerate safetensors sentencepiece protobuf scipy
hf = secret("HF_TOKEN")
if hf:
    os.environ["HF_TOKEN"] = hf

In [ ]:
# tag -> Hugging Face repo. Comment out any you do not want to rerun.
MODELS = {
    "tinyllama-1.1b-chat-v1.0": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "qwen2.5-7b-instruct": "Qwen/Qwen2.5-7B-Instruct",             # the T1 Qwen
    "qwen3-8b": "Qwen/Qwen3-8B",                                  # the original T3 Qwen
    "mistral-7b-instruct-v0.3": "mistralai/Mistral-7B-Instruct-v0.3",   # gated
    "llama3.1-8b-instruct": "meta-llama/Llama-3.1-8B-Instruct",        # gated
}
PRESET = "pi-robust"      # or "ci-rerun" for the full config grid
PRESETS = {
    # FP + 3-bit TurboQuant; the Pi-robust inverter trains on 8 random rotations
    "pi-robust": {"T3_CONFIGS": "fp16,turbo_k3v4", "T3_ROBUST_ROTATIONS": "8"},
    "ci-rerun": {"T3_ROBUST_ROTATIONS": "0"},
}
RUN_TAG = PRESET

In [ ]:
import shutil, subprocess, sys
from huggingface_hub import snapshot_download

for tag, repo in MODELS.items():
    out_dir = f"results/t3_leakage/{tag}-{RUN_TAG}"
    if os.path.exists(f"{out_dir}/leakage.json"):
        print("skip (done):", tag)
        continue
    local = f"/content/models/{tag}"
    print("\n=== downloading", repo)
    snapshot_download(repo, local_dir=local,
                      allow_patterns=["*.json", "*.safetensors", "tokenizer.model", "*.txt"])
    env = dict(os.environ, MODEL_ID=local, MODEL_TAG=f"{tag}-{RUN_TAG}", OUTPUT_DIR=out_dir,
               PYTHONUNBUFFERED="1", **PRESETS[PRESET])
    rc = subprocess.run([sys.executable, "t3_leakage/run_leakage.py"], env=env).returncode
    print("exit code", rc)
    shutil.rmtree(local, ignore_errors=True)   # free disk before the next model

In [ ]:
import glob, json
for path in sorted(glob.glob(f"results/t3_leakage/*-{RUN_TAG}/leakage.json")):
    d = json.load(open(path))
    print(f"\n{d['model_tag']}: verdict={d['verdict']} turboquant_specific={d['turboquant_specific']} "
          f"({d['texts']} texts, {d['vectors']} vectors, {d['elapsed_s']}s)")
    for layer in d["per_layer"]:
        print(f"  layer {layer['layer']}: {layer['verdict']}  turbo_k3v4 delta={layer.get('turbo_k3v4_public_delta')} "
              f"CI={layer.get('turbo_k3v4_public_ci95')}")
        for row in layer["results"]:
            if row["config"] == "turbo_k3v4":
                print(f"    {row['pi_regime']:<13} token={row['token_recovery']:.3f} delta_vs_fp={row['delta_vs_fp']:+.3f}")
        if "secret_pi" in layer:
            print("    secret_pi:", layer["secret_pi"])

In [ ]:
from google.colab import files
!cd results/t3_leakage && zip -q -r /content/t3_results.zip *-{RUN_TAG}
files.download("/content/t3_results.zip")